# Homework 9 - ST - 554 Big data analysis

## Author: Yefrid Cordoba

In this homework, we will model the compressive strength of concrete based on the amount of each component per cubic meter of material(concrete). The dataset *Concrete Compressive Strength* from the UC Irvine Machine Learning Repository will be used. We will fit three models: a multiple linear regression (MLR), a lasso regression, and a random forest — and compare their performance using the root mean squared error (RMSE) to determine the best model.

We initialize the sparl session.

In [17]:
!pip install pyspark

In [18]:
import pandas as pd
import numpy as np
from pyspark.ml.feature import SQLTransformer, StringIndexer, Binarizer, VectorAssembler, Interaction
from pyspark.ml.regression import LinearRegression, RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

In [9]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.master('local[*]').appName('my_app').getOrCreate()

### Splitting the data, metrics, and models

#### Split

The data is going to be imported as `pd.DataFrame`, and then turned into a sparkSQL dataframe

In [10]:
Concrete = pd.read_excel("Concrete_Data.xls", engine='xlrd')
Concrete = spark.createDataFrame(Concrete)
Concrete.columns

['Cement (component 1)(kg in a m^3 mixture)',
 'Blast Furnace Slag (component 2)(kg in a m^3 mixture)',
 'Fly Ash (component 3)(kg in a m^3 mixture)',
 'Water  (component 4)(kg in a m^3 mixture)',
 'Superplasticizer (component 5)(kg in a m^3 mixture)',
 'Coarse Aggregate  (component 6)(kg in a m^3 mixture)',
 'Fine Aggregate (component 7)(kg in a m^3 mixture)',
 'Age (day)',
 'Concrete compressive strength(MPa, megapascals) ']

Now we rename the columns

First we create a list with the new names for the variables, in this step we start preparing the column names for the use of MLlib which requires the response variable to be called `label`, while the vector with the explanatory variables or `features` is going to be created later.

In [11]:
names = ['cement', 'slag', 'ash', 'water', 'plasticizer', 'coarse', 'fine', 'age', 'strength']

In [12]:
Concrete = Concrete.toDF(*names)

In [13]:
Concrete.show(5)

+------+-----+---+-----+-----------+------+-----+---+------------------+
|cement| slag|ash|water|plasticizer|coarse| fine|age|          strength|
+------+-----+---+-----+-----------+------+-----+---+------------------+
| 540.0|  0.0|0.0|162.0|        2.5|1040.0|676.0| 28|       79.98611076|
| 540.0|  0.0|0.0|162.0|        2.5|1055.0|676.0| 28|61.887365759999994|
| 332.5|142.5|0.0|228.0|        0.0| 932.0|594.0|270|40.269535256000005|
| 332.5|142.5|0.0|228.0|        0.0| 932.0|594.0|365|      41.052779992|
| 198.6|132.4|0.0|192.0|        0.0| 978.4|825.5|360|      44.296075096|
+------+-----+---+-----+-----------+------+-----+---+------------------+
only showing top 5 rows


Here we are going to split the data between the training an test set for the models.

In [14]:
train, test = Concrete.randomSplit([0.8,0.2], seed = 12)

#### Metrics

The root mean squared error (`RMSE`) is the metric that will be used to judge all the models to predict the compressive strength of concrete.\
This metric is chosen because it penalizes larger deviations more heavily than smaller ones. `RMSE` is calculated by squaring the residuals, averaging them, and then taking the square root, a process that amplifies the influence of data points that deviate significantly from the predicted values, as shown in the formula below.

$
RMSE = \sqrt{\sum_{i=1}^n{\frac{(\hat{y}_i - y_i)^2}{n}}}
$

#### Models

##### MLR

The multiple linear regression is an extension of simple linear regression that account for the multivariate nature of data. Each of the coefficients $\beta_i$ represents how variable the response is with respect to their $x_i$, that is, with each of the betas.\
The loss function used to fit the model is the RMSE, which is also going to be the metric to compare and judge between the models.

$
y = \beta_0 + \beta_1x_1 + ... + \beta_kx_k + \epsilon
$

From this model, interaction terms will also be included in the evaluation to assess the fit of this model and compare it with regularized model (Lasso) and random forest and decision tree models.

##### Lasso

Lasso regression is a regularization technique that enhances linear regression by adding an $L_1$ (aka $\lambda$)penalty to the loss function. This penalty shrinks less important coefficients toward zero, effectively performing automated variable selection by zeroing out redundant or irrelevant predictors. This will reduce model complexity and prevent overfitting.

$
\sum_{i = 1}^n({y_i}-\sum_j{x_{ij}\beta_j})^2 + \lambda \sum_{j=1}^p{|\beta_j|}
$

##### Random forest

This method uses bootstrap aggregating and randomness to reduce the high variance from decision trees. Each tree is trained on a random sample of the data, ensuring the model sees a slightly different distribution. Also, the model only considers a random subset of predictors rather than the full set.\
The model aggregates the results, averaging the outputs for regression.

### Model fitting

#### MLR

In this model we are going to fit a model with two interaction terms `plasticizer*coarse` and `plasticizer*fine`, also we are going to create a transformation for the scaling of all of the variables before fitting the

In [ ]:
sqlTrans = SQLTransformer(
    statement = """
                SELECT year, log(km_driven) as log_km_driven, log(selling_price) as label FROM __THIS__
                """
)

After ranaming the columns, a column of features with the explanatory variables with be created, using the `VectorAssembler()` transformation method on the data.

In [15]:
assembler = VectorAssembler(inputCols = ['cement', 'slag', 'ash', 'water', 'plasticizer', 'coarse', 'fine', 'age'], \
                            outputCol = "features")

In [16]:
assembler.transform(Concrete).show(5)

+------+-----+---+-----+-----------+------+-----+---+------------------+--------------------+
|cement| slag|ash|water|plasticizer|coarse| fine|age|          strength|            features|
+------+-----+---+-----+-----------+------+-----+---+------------------+--------------------+
| 540.0|  0.0|0.0|162.0|        2.5|1040.0|676.0| 28|       79.98611076|[540.0,0.0,0.0,16...|
| 540.0|  0.0|0.0|162.0|        2.5|1055.0|676.0| 28|61.887365759999994|[540.0,0.0,0.0,16...|
| 332.5|142.5|0.0|228.0|        0.0| 932.0|594.0|270|40.269535256000005|[332.5,142.5,0.0,...|
| 332.5|142.5|0.0|228.0|        0.0| 932.0|594.0|365|      41.052779992|[332.5,142.5,0.0,...|
| 198.6|132.4|0.0|192.0|        0.0| 978.4|825.5|360|      44.296075096|[198.6,132.4,0.0,...|
+------+-----+---+-----+-----------+------+-----+---+------------------+--------------------+
only showing top 5 rows


In [19]:
assemb1 = VectorAssembler(inputCols=['plasticizer'], outputCol='pl_vec')
assemb1.transform(Concrete).show(5)

+------+-----+---+-----+-----------+------+-----+---+------------------+------+
|cement| slag|ash|water|plasticizer|coarse| fine|age|          strength|pl_vec|
+------+-----+---+-----+-----------+------+-----+---+------------------+------+
| 540.0|  0.0|0.0|162.0|        2.5|1040.0|676.0| 28|       79.98611076| [2.5]|
| 540.0|  0.0|0.0|162.0|        2.5|1055.0|676.0| 28|61.887365759999994| [2.5]|
| 332.5|142.5|0.0|228.0|        0.0| 932.0|594.0|270|40.269535256000005| [0.0]|
| 332.5|142.5|0.0|228.0|        0.0| 932.0|594.0|365|      41.052779992| [0.0]|
| 198.6|132.4|0.0|192.0|        0.0| 978.4|825.5|360|      44.296075096| [0.0]|
+------+-----+---+-----+-----------+------+-----+---+------------------+------+
only showing top 5 rows


Now that we have the data ready to be used in `MLlib` we can proceed to fit the